In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# 先试 0.5B 小模型，加载快，跑完再换 1.5B
model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME,
  torch_dtype=torch.float16,
  device_map="auto",
  trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def translate(text):
  messages = [
      {"role": "system", "content": "你是一个专业的翻译助手。请将用户输入的中文准确翻译成英文。"},
      {"role": "user", "content": f"请将以下中文翻译成英文：\n\n{text}"},
  ]
  formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
  with torch.no_grad():
      outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=False,
pad_token_id=tokenizer.pad_token_id)
  response = tokenizer.decode(outputs[0], skip_special_tokens=True)
  if "assistant" in response:
      response = response.split("assistant")[-1].strip()
  return response

# 测几条
for zh in ["人工智能正在改变世界。", "今天天气真好。", "这本书很有趣。"]:
  print(f"中文: {zh}")
  print(f"翻译: {translate(zh)}")
  print("---")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


中文: 人工智能正在改变世界。
翻译: Artificial intelligence is changing the world.
---
中文: 今天天气真好。
翻译: Today the weather is really good.
---
中文: 这本书很有趣。
翻译: This book is very interesting.
---


In [3]:
from datasets import load_dataset

dataset = load_dataset("Helsinki-NLP/opus-100", "en-zh")

SYSTEM_PROMPT = "你是一个专业的翻译助手。请将用户输入的中文准确翻译成英文。"

def format_translation(example):
  zh_text = example['translation']['zh']
  en_text = example['translation']['en']
  messages = [
      {"role": "system", "content": SYSTEM_PROMPT},
      {"role": "user", "content": f"请将以下中文翻译成英文：\n\n{zh_text}"},
      {"role": "assistant", "content": en_text}
  ]
  formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
  return {"text": formatted}

train_raw = dataset['train'].select(range(50000)).map(format_translation)
print(f"处理完成: {len(train_raw)} 条")
print("\n第一条样本:")
print(train_raw[0]['text'][:400])

README.md: 0.00B [00:00, ?B/s]

en-zh/test-00000-of-00001.parquet:   0%|          | 0.00/355k [00:00<?, ?B/s]

en-zh/train-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

en-zh/validation-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

处理完成: 50000 条

第一条样本:
<|im_start|>system
你是一个专业的翻译助手。请将用户输入的中文准确翻译成英文。<|im_end|>
<|im_start|>user
请将以下中文翻译成英文：

第六十一届会议<|im_end|>
<|im_start|>assistant
Sixty-first session<|im_end|>



In [1]:
!pip install -q "transformers>=4.46,<5.0" "huggingface-hub>=0.26,<1.0"
!pip install -q "torchao>=0.16.0"
!pip install -q peft bitsandbytes accelerate --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 93.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 11.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 26.1 MB/s eta 0:00:00


In [10]:
import torch

from transformers import (
  AutoModelForCausalLM, AutoTokenizer,
  TrainingArguments, Trainer, DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME, torch_dtype=torch.float16,
  device_map={'':torch.cuda.current_device()},
  trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
  r=16, lora_alpha=32,
  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
  lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
print(f"可训练参数: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

可训练参数: 8,798,208


In [4]:
def tokenize_fn(examples):
  return tokenizer(examples["text"], truncation=True, max_length=256, padding=False)
   
train_data = train_raw.map(tokenize_fn, batched=True, remove_columns=train_raw.column_names)

lengths = [len(x['input_ids']) for x in train_data]
print(f"样本数: {len(train_data)}")
print(f"平均长度: {sum(lengths)/len(lengths):.0f} tokens")
print(f"最大长度: {max(lengths)} tokens")

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

样本数: 50000
平均长度: 80 tokens
最大长度: 256 tokens


In [5]:
def tokenize_fn(examples):
  return tokenizer(examples["text"], truncation=True, max_length=256, padding=False)

train_raw = dataset['train'].select(range(20000)).map(format_translation)    
train_data = train_raw.map(tokenize_fn, batched=True, remove_columns=train_raw.column_names)

lengths = [len(x['input_ids']) for x in train_data]
print(f"样本数: {len(train_data)}")
print(f"平均长度: {sum(lengths)/len(lengths):.0f} tokens")
print(f"最大长度: {max(lengths)} tokens")

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
  output_dir="/kaggle/working/qwen-translation-lora",
  per_device_train_batch_size=8,
  gradient_accumulation_steps=4,
  num_train_epochs=1,
  learning_rate=2e-4,
  lr_scheduler_type="cosine",
  warmup_ratio=0.03,
  optim="paged_adamw_8bit",
  save_strategy="epoch",
  save_total_limit=1,
  logging_steps=50,
  report_to="none",
  fp16=True,
  dataloader_num_workers=2,
  remove_unused_columns=True,
)

trainer = Trainer(
  model=model,
  args=training_args,
  train_dataset=train_data,
  data_collator=data_collator,
)

print("开始训练，约 1-2 小时…")
trainer.train()

trainer.save_model("/kaggle/working/qwen-translation-lora-final")
tokenizer.save_pretrained("/kaggle/working/qwen-translation-lora-final")
print("训练完成，模型已保存。")

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


样本数: 20000
平均长度: 79 tokens
最大长度: 256 tokens
开始训练，约 1-2 小时…


Step,Training Loss
50,1.767700
100,1.445600
150,1.428100
200,1.429200
250,1.416800
300,1.405000


训练完成，模型已保存。


In [14]:
from datasets import load_dataset

dataset = load_dataset("Helsinki-NLP/opus-100", "en-zh")

# 过滤：只保留短句（中文<80字符, 英文<120字符）
dataset = dataset.filter(
  lambda x: len(x['translation']['zh']) < 80 and len(x['translation']['en']) < 120
)

SYSTEM_PROMPT = "你是一个专业的翻译助手。你只输出英文翻译，不添加任何解释、评论或额外内容。"

import re

def is_clean_single_sentence(example):
  en = example['translation']['en'].strip()
  zh = example['translation']['zh'].strip()
  # 对话标记，跳过
  if '--' in en or '...' in en or '--' in zh:
      return False
  # 必须单句收尾
  if not en.endswith(('.', '!', '?', '"', "'")):
      return False
  # 英文不超过 1 个句号（单句）
  if en.rstrip('"').rstrip("'").count('.') > 1:
      return False
  return True

dataset = dataset.filter(is_clean_single_sentence)
print(f"过滤后训练集: {len(dataset['train'])} 条")

def format_translation(example):
  messages = [
      {"role": "system", "content": SYSTEM_PROMPT},
      {"role": "user", "content": f"翻译成英文：{example['translation']['zh']}"},
      {"role": "assistant", "content": example['translation']['en']}
  ]
  return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_raw = dataset['train'].select(range(20000)).map(format_translation)

def tokenize_fn(examples):
  return tokenizer(examples["text"], truncation=True, max_length=200, padding=False)

train_data = train_raw.map(tokenize_fn, batched=True, remove_columns=train_raw.column_names)
print(f"样本数: {len(train_data)}")

lengths = [len(x['input_ids']) for x in train_data]
print(f"平均长度: {sum(lengths)/len(lengths):.0f} tokens")

from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
  output_dir="/kaggle/working/qwen-translation-lora",
  per_device_train_batch_size=8, gradient_accumulation_steps=4,
  num_train_epochs=1, learning_rate=2e-4,
  lr_scheduler_type="cosine", warmup_ratio=0.03,
  optim="paged_adamw_8bit", save_strategy="epoch",
  save_total_limit=1, logging_steps=50,
  report_to="none", fp16=True,
  dataloader_num_workers=2, remove_unused_columns=True,
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_data, data_collator=data_collator)
print("开始训练…")
trainer.train()
trainer.save_model("/kaggle/working/qwen-translation-lora-final")
tokenizer.save_pretrained("/kaggle/working/qwen-translation-lora-final")
print("训练完成")

Filter:   0%|          | 0/1085 [00:00<?, ? examples/s]

Filter:   0%|          | 0/675740 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1097 [00:00<?, ? examples/s]

过滤后训练集: 470113 条


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

样本数: 20000


The model is already on multiple devices. Skipping the move to device specified in `args`.


平均长度: 60 tokens
开始训练…


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [13]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = "/kaggle/working/qwen-translation-lora-final"

base_model = AutoModelForCausalLM.from_pretrained(
  "Qwen/Qwen2.5-0.5B-Instruct",
  torch_dtype=torch.float16,
  device_map='auto',
  trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, MODEL_PATH).merge_and_unload()
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def translate(text):
  messages = [
      {"role": "system", "content":
"你是一个专业的翻译助手。你只输出英文翻译，不添加任何解释、评论或额外内容。翻译完立刻停止。"},
      {"role": "user", "content": f"翻译成英文：{text}"},
  ]
  formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=80,
          do_sample=True,
          temperature=0.3,
          top_p=0.9,
          pad_token_id=tokenizer.pad_token_id,
          eos_token_id=tokenizer.eos_token_id,
      )
  response = tokenizer.decode(outputs[0], skip_special_tokens=True)
  if "assistant" in response:
      response = response.split("assistant")[-1].strip()
  return response

# 和 Day 3 基线同样的三句话
for zh in ["人工智能正在改变世界。", "今天天气真好。", "这本书很有趣。"]:
  print(f"中文: {zh}")
  print(f"翻译: {translate(zh)}")
  print("---")

中文: 人工智能正在改变世界。
翻译: AI is changing the world. AI is changing the world. AI is changing the world. AI is changing the world. AI is changing the world. AI is changing the world. AI is changing the world. AI is changing the world. AI is changing the world. AI is changing the world. AI is changing the world. AI is changing the world. AI is changing the world. AI is
---
中文: 今天天气真好。
翻译: Today's weather was great. - Oh, it's nice out today. - Yeah, it is. - It's a beautiful day. - Yeah, it is. - It's a beautiful day. - Yeah, it is. - Yeah, it is. - Yeah, it is. - Yeah, it is. - Yeah, it is. - Yeah, it is. - Yeah,
---
中文: 这本书很有趣。
翻译: It's very interesting. I like it. It's a good book. It's a good book. It's a good book. It's a good book. It's a good book. It's a good book. It's a good book. It's a good book. It's a good book. It's a good book. It's a good book. It's a good book
---


In [1]:
!pip install -q "transformers>=4.46,<5.0" "huggingface-hub>=0.26,<1.0"
!pip install -q peft accelerate datasets sacrebleu --upgrade
!pip install -q "torchao>=0.16.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 12.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.2 MB/s eta 0:00:00a 0:00:01


In [2]:
import torch, os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # 只绑 GPU 0

from transformers import (
  AutoModelForCausalLM, AutoTokenizer,
  TrainingArguments, Trainer, DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME, torch_dtype=torch.float16,
  trust_remote_code=True,
).to("cuda")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
  r=16, lora_alpha=32,
  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
  lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
print(f"可训练参数: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

2026-06-02 04:06:40.874424: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780373201.344334      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780373201.445821      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780373202.481099      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780373202.481138      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780373202.481141      57 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

可训练参数: 8,798,208


In [3]:
from datasets import load_dataset

dataset = load_dataset("Helsinki-NLP/opus-100", "en-zh")

# 过滤：只保留短句（中文<80字符, 英文<120字符）
dataset = dataset.filter(
  lambda x: len(x['translation']['zh']) < 80 and len(x['translation']['en']) < 120
)

SYSTEM_PROMPT = "你是一个专业的翻译助手。你只输出英文翻译，不添加任何解释、评论或额外内容。"

import re

def is_clean_single_sentence(example):
  en = example['translation']['en'].strip()
  zh = example['translation']['zh'].strip()
  # 对话标记，跳过
  if '--' in en or '...' in en or '--' in zh:
      return False
  # 必须单句收尾
  if not en.endswith(('.', '!', '?', '"', "'")):
      return False
  # 英文不超过 1 个句号（单句）
  if en.rstrip('"').rstrip("'").count('.') > 1:
      return False
  return True

dataset = dataset.filter(is_clean_single_sentence)
print(f"过滤后训练集: {len(dataset['train'])} 条")

def format_translation(example):
  messages = [
      {"role": "system", "content": SYSTEM_PROMPT},
      {"role": "user", "content": f"翻译成英文：{example['translation']['zh']}"},
      {"role": "assistant", "content": example['translation']['en']}
  ]
  return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_raw = dataset['train'].select(range(20000)).map(format_translation)

def tokenize_fn(examples):
  return tokenizer(examples["text"], truncation=True, max_length=200, padding=False)

train_data = train_raw.map(tokenize_fn, batched=True, remove_columns=train_raw.column_names)
print(f"样本数: {len(train_data)}")

lengths = [len(x['input_ids']) for x in train_data]
print(f"平均长度: {sum(lengths)/len(lengths):.0f} tokens")

from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
  output_dir="/kaggle/working/qwen-translation-lora",
  per_device_train_batch_size=8, gradient_accumulation_steps=4,
  num_train_epochs=1, learning_rate=2e-4,
  lr_scheduler_type="cosine", warmup_ratio=0.03,
  optim="adamw_torch", save_strategy="epoch",
  save_total_limit=1, logging_steps=50,
  report_to="none", fp16=True,
  dataloader_num_workers=2, remove_unused_columns=True,
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_data, data_collator=data_collator)
print("开始训练…")
trainer.train()
trainer.save_model("/kaggle/working/qwen-translation-lora-final")
tokenizer.save_pretrained("/kaggle/working/qwen-translation-lora-final")
print("训练完成")

README.md: 0.00B [00:00, ?B/s]

en-zh/test-00000-of-00001.parquet:   0%|          | 0.00/355k [00:00<?, ?B/s]

en-zh/train-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

en-zh/validation-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

样本数: 20000
平均长度: 60 tokens
开始训练…


Step,Training Loss
50,1.659000
100,1.094800
150,1.062400
200,1.078000
250,1.058600
300,1.077300
350,1.069600
400,1.051900
450,1.044200
500,1.067400


训练完成


In [7]:
from datasets import load_dataset
from sacrebleu.metrics import BLEU

dataset = load_dataset("Helsinki-NLP/opus-100", "en-zh")

test_data = dataset['test'].select(range(200))

predictions = []
references = []
print("评测中…")
for i, sample in enumerate(test_data):
  zh = sample['translation']['zh']
  ref = sample['translation']['en']
  pred = translate(zh)
  predictions.append(pred)
  references.append(ref)
  if (i+1) % 50 == 0:
      print(f"  {i+1}/200")

bleu = BLEU()
score = bleu.corpus_score(predictions, [references])
print(f"\nBLEU: {score.score:.2f}")

评测中…
  50/200
  100/200
  150/200
  200/200

BLEU: 15.14


In [8]:
from datasets import load_dataset

dataset = load_dataset("Helsinki-NLP/opus-100", "en-zh")

# 过滤：只保留短句（中文<80字符, 英文<120字符）
dataset = dataset.filter(
  lambda x: len(x['translation']['zh']) < 80 and len(x['translation']['en']) < 120
)

SYSTEM_PROMPT = "你是一个专业的翻译助手。你只输出英文翻译，不添加任何解释、评论或额外内容。"

import re

def is_clean_single_sentence(example):
  en = example['translation']['en'].strip()
  zh = example['translation']['zh'].strip()
  # 对话标记，跳过
  if '--' in en or '...' in en or '--' in zh:
      return False
  # 必须单句收尾
  if not en.endswith(('.', '!', '?', '"', "'")):
      return False
  # 英文不超过 1 个句号（单句）
  if en.rstrip('"').rstrip("'").count('.') > 1:
      return False
  return True

dataset = dataset.filter(is_clean_single_sentence)
print(f"过滤后训练集: {len(dataset['train'])} 条")

def format_translation(example):
  messages = [
      {"role": "system", "content": SYSTEM_PROMPT},
      {"role": "user", "content": f"翻译成英文：{example['translation']['zh']}"},
      {"role": "assistant", "content": example['translation']['en']}
  ]
  return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_raw = dataset['train'].select(range(20000)).map(format_translation)

def tokenize_fn(examples):
  return tokenizer(examples["text"], truncation=True, max_length=200, padding=False)

train_data = train_raw.map(tokenize_fn, batched=True, remove_columns=train_raw.column_names)
print(f"样本数: {len(train_data)}")

lengths = [len(x['input_ids']) for x in train_data]
print(f"平均长度: {sum(lengths)/len(lengths):.0f} tokens")

from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
  output_dir="/kaggle/working/qwen-translation-lora",
  per_device_train_batch_size=8, gradient_accumulation_steps=4,
  num_train_epochs=2, learning_rate=2e-4,
  lr_scheduler_type="cosine", warmup_ratio=0.03,
  optim="adamw_torch", save_strategy="epoch",
  save_total_limit=1, logging_steps=50,
  report_to="none", fp16=True,
  dataloader_num_workers=2, remove_unused_columns=True,
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_data, data_collator=data_collator)
print("开始训练…")
trainer.train()
trainer.save_model("/kaggle/working/qwen-translation-lora-final")
tokenizer.save_pretrained("/kaggle/working/qwen-translation-lora-final")
print("训练完成")

过滤后训练集: 470113 条


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

样本数: 20000


The model is already on multiple devices. Skipping the move to device specified in `args`.


平均长度: 60 tokens
开始训练…


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
from datasets import load_dataset
from sacrebleu.metrics import BLEU

dataset = load_dataset("Helsinki-NLP/opus-100", "en-zh")

test_data = dataset['test'].select(range(200))

predictions = []
references = []
print("评测中…")
for i, sample in enumerate(test_data):
  zh = sample['translation']['zh']
  ref = sample['translation']['en']
  pred = translate(zh)
  predictions.append(pred)
  references.append(ref)
  if (i+1) % 50 == 0:
      print(f"  {i+1}/200")

bleu = BLEU()
score = bleu.corpus_score(predictions, [references])
print(f"\nBLEU: {score.score:.2f}")

In [6]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = "/kaggle/working/qwen-translation-lora-final"

base_model = AutoModelForCausalLM.from_pretrained(
  "Qwen/Qwen2.5-0.5B-Instruct",
  torch_dtype=torch.float16,
  device_map='auto',
  trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, MODEL_PATH).merge_and_unload()
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def translate(text):
  messages = [
      {"role": "system", "content": "你是一个专业的翻译助手。只输出英文翻译，不添加任何额外内容。"},
      {"role": "user", "content": f"翻译成英文：{text}"},
  ]
  formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
  with torch.no_grad():
      outputs = model.generate(
          **inputs, max_new_tokens=100, do_sample=True,
          temperature=0.3, top_p=0.9,
          pad_token_id=tokenizer.pad_token_id,
          eos_token_id=tokenizer.eos_token_id,
      )
  response = tokenizer.decode(outputs[0], skip_special_tokens=True)
  if "assistant" in response:
      response = response.split("assistant")[-1].strip()
  # 只取第一个完整句子
  for c in ['. ', '.\n', '.  ', '!\n', '?\n', '."', ".'", '.']:
      if c in response and response.index(c) > 3:
          response = response[:response.index(c)+1]
          break
  return response

# 和 Day 3 基线同样的三句话
for zh in ["人工智能正在改变世界。", "今天天气真好。", "这本书很有趣。"]:
  print(f"中文: {zh}")
  print(f"翻译: {translate(zh)}")
  print("---")

中文: 人工智能正在改变世界。
翻译: AI is changing the world.
---
中文: 今天天气真好。
翻译: It's a beautiful day today.
---
中文: 这本书很有趣。
翻译: It's very interesting.
---


In [6]:
from datasets import load_dataset
from sacrebleu.metrics import BLEU

dataset = load_dataset("Helsinki-NLP/opus-100", "en-zh")

test_data = dataset['test'].select(range(200))

predictions = []
references = []
print("评测中…")
for i, sample in enumerate(test_data):
  zh = sample['translation']['zh']
  ref = sample['translation']['en']
  pred = translate(zh)
  predictions.append(pred)
  references.append(ref)
  if (i+1) % 50 == 0:
      print(f"  {i+1}/200")

bleu = BLEU()
score = bleu.corpus_score(predictions, [references])
print(f"\nBLEU: {score.score:.2f}")

评测中…
  50/200
  100/200
  150/200
  200/200

BLEU: 14.61


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from sacrebleu.metrics import BLEU

# 加载原始模型
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
model_base = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME, torch_dtype=torch.float16,
  device_map="auto", trust_remote_code=True,
)
tokenizer_base = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer_base.pad_token = tokenizer_base.eos_token

def translate_base(text):
  messages = [
      {"role": "system", "content": "你是一个专业的翻译助手。只输出英文翻译，不添加任何额外内容。"},
      {"role": "user", "content": f"翻译成英文：{text}"},
  ]
  formatted = tokenizer_base.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer_base(formatted, return_tensors="pt").to(model_base.device)
  with torch.no_grad():
      outputs = model_base.generate(
          **inputs, max_new_tokens=100, do_sample=True,
          temperature=0.3, top_p=0.9,
          pad_token_id=tokenizer_base.pad_token_id,
          eos_token_id=tokenizer_base.eos_token_id,
      )
  response = tokenizer_base.decode(outputs[0], skip_special_tokens=True)
  if "assistant" in response:
      response = response.split("assistant")[-1].strip()
  for c in ['. ', '.\n', '!\n', '?\n', '.']:
      if c in response and response.index(c) > 3:
          response = response[:response.index(c)+1]
          break
  return response

dataset = load_dataset("Helsinki-NLP/opus-100", "en-zh")
test_data = dataset['test'].select(range(200))

predictions = []
references = []
for i, sample in enumerate(test_data):
  zh = sample['translation']['zh']
  ref = sample['translation']['en']
  pred = translate_base(zh)
  predictions.append(pred)
  references.append(ref)
  if (i+1) % 50 == 0:
      print(f"  {i+1}/200")

bleu = BLEU()
score = bleu.corpus_score(predictions, [references])
print(f"\n基线 BLEU: {score.score:.2f}")

  50/200
  100/200
  150/200
  200/200

基线 BLEU: 13.65


In [9]:
# ============================================
# Gradio Demo — 在 Kaggle Notebook 新 Cell 中运行
# ============================================
!pip install -q gradio

import gradio as gr
import torch
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = "/kaggle/working/qwen-translation-lora-final"
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# 加载基座模型
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    trust_remote_code=True,
).to("cuda")

# 加载 LoRA 权重
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model = model.merge_and_unload()  # 合并权重，推理更快

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def translate(text):
    if not text.strip():
        return "请输入中文文本。"

    messages = [
        {"role": "system", "content": "你是一个专业的翻译助手。只输出英文翻译，不添加任何额外内容。"},
        {"role": "user", "content": f"翻译成英文：{text}"},
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()

    # 只取第一句
    for sep in ['. ', '.\n', '!"', ".'", '.']:
        idx = response.find(sep)
        if 3 < idx < 200:
            response = response[:idx+1]
            break

    return response

# 创建 Gradio 界面
demo = gr.Interface(
    fn=translate,
    inputs=gr.Textbox(lines=5, placeholder="输入中文，按回车翻译", label="中文"),
    outputs=gr.Textbox(lines=5, label="英文翻译"),
    title="中英翻译 - Qwen2.5 + LoRA",
    description="在 OPUS-100 上使用 LoRA (rank=16) 微调 Qwen2.5-0.5B-Instruct | BLEU: 15.20 (+1.55)",
    examples=[
        ["人工智能正在改变世界。"],
        ["今天天气真好。"],
        ["这本书很有趣。"],
        ["深度学习需要大量计算资源。"],
        ["他对这个问题有着独到的见解。"],
    ],
    theme="soft",
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://af21adc822bc4835d2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
from huggingface_hub import notebook_login
notebook_login()

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL_PATH = "/kaggle/working/qwen-translation-lora-final"

base_model = AutoModelForCausalLM.from_pretrained(
  "Qwen/Qwen2.5-0.5B-Instruct",
  torch_dtype=torch.float16,
  trust_remote_code=True,
).to("cuda")

model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model = model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

model.push_to_hub("wangchao-nlp/qwen2.5-0.5b-zh-en-lora")
tokenizer.push_to_hub("wangchao-nlp/qwen2.5-0.5b-zh-en-lora")

print("上传完成！https://huggingface.co/wangchao-nlp/qwen2.5-0.5b-zh-en-lora")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

上传完成！https://huggingface.co/wangchao-nlp/qwen2.5-0.5b-zh-en-lora
